In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [3]:


# Start Spark
spark = (
    SparkSession.builder
    .appName("chris-jupyter-notebook")
    .getOrCreate()
)


print("Spark started")

# Path
CHUNKS_DIR = "../data/processed/chunks"
print("Files in chunks folder:", os.listdir(CHUNKS_DIR))

chunk_filename = "num_2020_chunk.csv" 
chunk_path = os.path.join(CHUNKS_DIR, chunk_filename)

# Load chunk as DataFrame
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(chunk_path)
)

print("Loaded:", chunk_path)
print("Row count:", df.count())
df.show(5, truncate=False)
df.printSchema()


Spark started
Files in chunks folder: ['sub_2020_chunk.csv', 'pre_2020_chunk.csv', 'tag_2020_chunk.csv', 'num_2020_chunk.csv']
Loaded: ../data/processed/chunks/num_2020_chunk.csv
Row count: 10000
+--------------------+---------------------------------------------------+------------+--------+----+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+---------+--------+-------+----+
|adsh                |tag                                                |version     |ddate   |qtrs|uom|segments                                                                                                                                                             |coreg                 |value    |footnote|quarter|year|
+--------------------+---------------------------------------------------+------------+--------+----+---+---------------------------------------------

In [4]:

# Column names for nums file (num_2020_chunk.csv)
TAG_COL     = "tag"    # XBRL tag
COMPANY_COL = "adsh"   # filing ID (acts like entity/filing)
DATE_COL    = "ddate"  # date in YYYYMMDD format

# Q1: Which tags appear most often in this subset?
q1 = (
    df.groupBy(TAG_COL)
      .agg(F.count("*").alias("num_rows"))
      .orderBy(F.desc("num_rows"))
)

print("=== Q1: rows by tag ===")
q1.show(20, truncate=False)

# Q2: How many unique filings, and which filings have the most rows?
q2_unique = df.select(COMPANY_COL).distinct().count()
print(f"\n=== Q2: unique filings (adsh) ===\nUnique IDs: {q2_unique}")

q2_top = (
    df.groupBy(COMPANY_COL)
      .agg(F.count("*").alias("num_rows"))
      .orderBy(F.desc("num_rows"))
)

print("\nTop filings by number of rows:")
q2_top.show(20, truncate=False)

# Q3: How many rows per year (using ddate)?
df_with_year = df.withColumn(
    "year",
    F.year(
        F.to_date(
            F.col(DATE_COL).cast("string"),
            "yyyyMMdd"
        )
    )
)

q3 = (
    df_with_year.groupBy("year")
                .agg(F.count("*").alias("num_rows"))
                .orderBy("year")
)

print("\n=== Q3: rows per year ===")
q3.show(50, truncate=False)


=== Q1: rows by tag ===
+------------------------------------------------------------------------------------------------+--------+
|tag                                                                                             |num_rows|
+------------------------------------------------------------------------------------------------+--------+
|RevenueFromContractWithCustomerExcludingAssessedTax                                             |277     |
|StockholdersEquity                                                                              |252     |
|StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest                          |206     |
|NetIncomeLoss                                                                                   |191     |
|Revenues                                                                                        |164     |
|Assets                                                                                          |146     |
|Ope